In [13]:
import pandas as pd
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS

In [14]:
# --- CONFIGURACIÓN DE INFLUXDB (v2) ---
env_file = find_dotenv(".env", usecwd=True)
if not env_file:
   for candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"):
       if candidate.is_file():
           env_file = str(candidate)
           break


if not env_file:
   raise FileNotFoundError("No se encontró el archivo .env")


load_dotenv(env_file, override=True)


def required_env(name):
   value = os.getenv(name)
   if not value:
       raise RuntimeError(f"Falta la variable {name} en {env_file}")
   return value


INFLUX_URL = required_env("INFLUX_URL")
INFLUX_TOKEN = required_env("INFLUX_TOKEN")
INFLUX_ORG = required_env("INFLUX_ORG")
INFLUX_BUCKET_BAMBU = required_env("INFLUX_BUCKET_BAMBU")
# Busca automáticamente el archivo .env en el directorio actual

In [15]:


# ==============================================================================
# 1. CONFIGURACIÓN DE CONEXIÓN A INFLUXDB (v2)
# ==============================================================================

# Archivo CSV generado durante las impresiones
CSV_FILE = "../datos/dataset_bambu_multi_printer.csv"


# ==============================================================================
# 2. FUNCIÓN PRINCIPAL DE MIGRACIÓN
# ==============================================================================
def importar_csv_a_influx():
    """
    Lee el dataset local en CSV y mapea cada fila al protocolo de datos de InfluxDB
    organizando la información en Tags (metadatos de filtrado) y Fields (métricas).
    """
    print(f"Lectura del archivo CSV: '{CSV_FILE}'...")
    try:
        df = pd.read_csv(CSV_FILE)
    except FileNotFoundError:
        print(f"Error: No se encuentra el archivo '{CSV_FILE}'. Verifica la ruta.")
        return

    # Convertir la columna ISO-8601 de texto a objetos datetime nativos
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    # Inicializar el cliente oficial de InfluxDB y la API de escritura síncrona
    client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
    write_api = client.write_api(write_options=SYNCHRONOUS)

    puntos_batch = []
    total_filas = len(df)
    tamanio_lote = 500  # Agrupar lecturas para minimizar la latencia de red

    print(f"Subiendo {total_filas} registros al bucket '{INFLUX_BUCKET_BAMBU}'...")

    for index, row in df.iterrows():
        # Construcción del registro mediante el Line Protocol de InfluxDB
        punto = (
            Point("bambu_telemetry")  # Measurement: equivalente al nombre de la tabla
            
            # --- TAGS (Cadenas de texto indexadas para agrupar y filtrar rápidamente) ---
            .tag("printer_name", str(row["printer_name"]))    # Tag 1: Nombre de la máquina
            .tag("serial_number", str(row["serial_number"]))  # Tag 2: Número de serie único
            .tag("tray_type", str(row["tray_type"]))          # Material cargado desde el AMS
            .tag("gcode_state", str(row["gcode_state"]))      # Estado de ejecución (RUNNING, etc.)
            
            # --- FIELDS (Métricas numéricas sobre las que hacer gráficas o cálculos) ---
            .field("nozzle_temp", float(row["nozzle_temp"]))
            .field("bed_temp", float(row["bed_temp"]))
            .field("mc_percent", int(row["mc_percent"]))
            .field("layer_num", int(row["layer_num"]))
            .field("total_layer_num", int(row["total_layer_num"]))
            
            # --- TIMESTAMP (Estampa de tiempo precisa en nanosegundos) ---
            .time(row["timestamp"], WritePrecision.NS)
        )
        puntos_batch.append(punto)

        # Enviar el bloque a InfluxDB cada vez que alcance el tamaño límite
        if len(puntos_batch) >= tamanio_lote:
            write_api.write(bucket=INFLUX_BUCKET_BAMBU, org=INFLUX_ORG, record=puntos_batch)
            puntos_batch = []
            print(f"Procesadas {index + 1}/{total_filas} filas...")

    # Enviar los puntos restantes que no hayan completado el último lote
    if puntos_batch:
        write_api.write(bucket=INFLUX_BUCKET_BAMBU, org=INFLUX_ORG, record=puntos_batch)

    print(f"\nImportación finalizada. {total_filas} registros cargados correctamente en InfluxDB.")
    
    # Cerrar la conexión limpia con el servidor
    client.close()


# ==============================================================================
# 3. PUNTO DE ENTRADA
# ==============================================================================
if __name__ == "__main__":
    importar_csv_a_influx()

Lectura del archivo CSV: '../datos/dataset_bambu_multi_printer.csv'...
Subiendo 47227 registros al bucket 'bambu_maintenance'...
Procesadas 500/47227 filas...
Procesadas 1000/47227 filas...
Procesadas 1500/47227 filas...
Procesadas 2000/47227 filas...
Procesadas 2500/47227 filas...
Procesadas 3000/47227 filas...
Procesadas 3500/47227 filas...
Procesadas 4000/47227 filas...
Procesadas 4500/47227 filas...
Procesadas 5000/47227 filas...
Procesadas 5500/47227 filas...
Procesadas 6000/47227 filas...
Procesadas 6500/47227 filas...
Procesadas 7000/47227 filas...
Procesadas 7500/47227 filas...
Procesadas 8000/47227 filas...
Procesadas 8500/47227 filas...
Procesadas 9000/47227 filas...
Procesadas 9500/47227 filas...
Procesadas 10000/47227 filas...
Procesadas 10500/47227 filas...
Procesadas 11000/47227 filas...
Procesadas 11500/47227 filas...
Procesadas 12000/47227 filas...
Procesadas 12500/47227 filas...
Procesadas 13000/47227 filas...
Procesadas 13500/47227 filas...
Procesadas 14000/47227 fila